# 17 — Context tree prototype: the relational LUT-view and the algebraic S(t)

Prototype for **LUT_VIEW.md §7 (Design v2)**. It demonstrates:

1. **The relational view** — $V_j(H_i) = M(H_i, L_j)$ with provenance `(h, l)`;
2. **The full image** — $V(H_i) = ∪_j V_j(H_i)$ over the LUT family;
3. **The context tree** — a Merkle tree over the working set of HLLSets,
   with each leaf carrying its per-LUT views;
4. **The algebraic S(t)** — $H(t) = (S(t), H(t-1), D, R, N)$ where $S(t)$ is
   the tree's union, and D/R/N are computed from it;
5. **The ewm-git link** — commits pin S(t); the tree root links the working
   set to its views.


In [2]:
:dep ewm-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-fpga-bridge/crates/ewm-core" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/hllset-core" }
:dep ewm-git = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/ewm-git" }
:dep lut-view = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/lut-view" }
:dep sha1 = "0.10"
:dep hex = "0.4"

In [3]:
use ewm_core::{parse_token_id, slice_positions_with, token_in_bytes, InLut, TokenSet};
use ewm_git::{view, LatticeState, MemoryStore, ObjectId, Repository};
use hllset_core::HLLSet;
use hllset_core::core::content_addr::{content_key_from_tokens, view_key_from_tokens};
use sha1::{Digest, Sha1};

fn tid(n: u32) -> Vec<u8> { token_in_bytes(n) }
fn sha1_hex(data: &[u8]) -> String { hex::encode(Sha1::digest(data)) }
fn h_key(ids: &[u32]) -> String {
    let toks: Vec<Vec<u8>> = ids.iter().map(|&n| tid(n)).collect();
    content_key_from_tokens(toks.iter())
}
fn v_key(ids: &[u32]) -> String {
    let toks: Vec<Vec<u8>> = ids.iter().map(|&n| tid(n)).collect();
    view_key_from_tokens(toks.iter())
}
fn node_hash(left: &str, right: &str) -> String {
    let mut data = vec![0x01u8];
    data.extend_from_slice(left.as_bytes());
    data.extend_from_slice(right.as_bytes());
    sha1_hex(&data)
}
println!("crates loaded; helpers ready");


crates loaded; helpers ready


---
## 1. The relational view: V_j(H_i) = M(H_i, L_j)

Two demo LUTs (token-space reverse indexes) and three HLLSets. The view
record carries **both** addresses: `h` (which HLLSet) and `l` (which LUT).


In [4]:
let lut_main: InLut = {
    let mut l = InLut::new();
    for n in 0..8 { l.insert(tid(n)); }
    l
};
let lut_extra: InLut = {
    let mut l = InLut::new();
    for n in 3..10 { l.insert(tid(n)); }
    l
};

let ha: HLLSet = HLLSet::from_tokens([tid(0), tid(2), tid(5), tid(8)]);
let hb: HLLSet = HLLSet::from_tokens([tid(1), tid(4)]);
let hc: HLLSet = HLLSet::from_tokens([tid(3), tid(8), tid(12)]);
println!("H_a bits={}  H_b bits={}  H_c bits={}", ha.popcount(), hb.popcount(), hc.popcount());


H_a bits=4  H_b bits=2  H_c bits=3


In [5]:
#[derive(Clone, Debug)]
struct ViewRecord { h: String, l: String, tokens: Vec<u32>, key: String }

fn materialize(hset: &HLLSet, lut: &InLut, l: &str, h_ids: &[u32]) -> ViewRecord {
    let positions: Vec<(u32, u32)> = hset.active_positions();
    let tokens: Vec<u32> = slice_positions_with(&positions, lut, parse_token_id).ids;
    ViewRecord { h: h_key(h_ids), l: l.to_string(), key: v_key(&tokens), tokens }
}

let v_main_ha = materialize(&ha, &lut_main, "main", &[0, 2, 5, 8]);
let v_extra_ha = materialize(&ha, &lut_extra, "extra", &[0, 2, 5, 8]);
println!("V_main(H_a)  = {:?}  {}", v_main_ha.tokens, v_main_ha.key);
println!("V_extra(H_a) = {:?}  {}", v_extra_ha.tokens, v_extra_ha.key);

let full_image: Vec<u32> = TokenSet::new(v_main_ha.tokens.clone())
    .union(&TokenSet::new(v_extra_ha.tokens.clone()))
    .ids;
println!("V(H_a) = union over LUTs = {:?}  {}", full_image, v_key(&full_image));


V_main(H_a)  = [0, 2, 5]  v:c49d544ec03cbf175da8a2a605020b3a7f7368ad
V_extra(H_a) = [5, 8]  v:f305ccc6a1923eb8c4cc70c5bedfd0bb9b7e1f72
V(H_a) = union over LUTs = [0, 2, 5, 8]  v:3eb559c1da4945697c0df135a0f6779a62a9ca05


---
## 2. The context tree: HLLSets as leaves, views hanging off them

Leaves are sorted by `h:<sha1>`; internal nodes are `sha1(0x01 ‖ left ‖ right)`.
The root is the **exact collection identity** of the working set — add or
remove a leaf and the root changes; diff two roots to get leaf-level D/R/N.


In [6]:
#[derive(Clone, Debug)]
struct Leaf { h: String, views: Vec<(String, String)> } // (lut, v:<sha1>)

#[derive(Clone, Debug)]
struct ContextTree { root: String, leaves: Vec<Leaf> }

fn tree_root(leaves: &[Leaf]) -> String {
    if leaves.is_empty() { return sha1_hex(b"empty-context"); }
    let mut level: Vec<String> = leaves.iter().map(|lf| {
        let mut data = vec![0x00u8];
        data.extend_from_slice(lf.h.as_bytes());
        for (l, v) in &lf.views {
            data.push(0u8);
            data.extend_from_slice(l.as_bytes());
            data.extend_from_slice(v.as_bytes());
        }
        sha1_hex(&data)
    }).collect();
    while level.len() > 1 {
        let mut next: Vec<String> = Vec::new();
        for pair in level.chunks(2) {
            let right = pair.get(1).cloned().unwrap_or_else(|| pair[0].clone());
            next.push(node_hash(&pair[0], &right));
        }
        level = next;
    }
    level[0].clone()
}

impl ContextTree {
    fn build(mut leaves: Vec<Leaf>) -> Self {
        leaves.sort_by(|a, b| a.h.cmp(&b.h));
        let root = tree_root(&leaves);
        Self { root, leaves }
    }
    fn root(&self) -> &str { &self.root }
    fn diff(&self, other: &Self) -> (Vec<String>, Vec<String>) {
        let mine: Vec<&str> = self.leaves.iter().map(|l| l.h.as_str()).collect();
        let theirs: Vec<&str> = other.leaves.iter().map(|l| l.h.as_str()).collect();
        let added: Vec<String> = theirs.iter().filter(|h| !mine.contains(h)).map(|s| s.to_string()).collect();
        let removed: Vec<String> = mine.iter().filter(|h| !theirs.contains(h)).map(|s| s.to_string()).collect();
        (added, removed)
    }
}


In [7]:
fn leaf_of(hset: &HLLSet, ids: &[u32], luts: &[(&str, &InLut)]) -> Leaf {
    let views: Vec<(String, String)> = luts.iter().map(|(name, lut)| {
        let r = materialize(hset, lut, name, ids);
        (name.to_string(), r.key)
    }).collect();
    Leaf { h: h_key(ids), views }
}

let leaves_prev: Vec<Leaf> = vec![
    leaf_of(&ha, &[0, 2, 5, 8], &[("main", &lut_main), ("extra", &lut_extra)]),
    leaf_of(&hb, &[1, 4], &[("main", &lut_main), ("extra", &lut_extra)]),
];
let leaves_now: Vec<Leaf> = vec![
    leaf_of(&ha, &[0, 2, 5, 8], &[("main", &lut_main), ("extra", &lut_extra)]),
    leaf_of(&hc, &[3, 8, 12], &[("main", &lut_main), ("extra", &lut_extra)]),
];

let tree_prev: ContextTree = ContextTree::build(leaves_prev);
let tree_now: ContextTree = ContextTree::build(leaves_now);
println!("S(t-1) tree root = {}", tree_prev.root());
println!("S(t)   tree root = {}", tree_now.root());
let (added, removed) = tree_now.diff(&tree_prev);
println!("tree diff: added={:?}  removed={:?}", added, removed);


S(t-1) tree root = 3abc82fc9e6b0f0457221a69083b999862952270
S(t)   tree root = c3eeb36c9e16e6f0efa8483c58c016db2ac86425
tree diff: added=["h:4980dd9b9e6f388b506cc209d37a3766984f779c"]  removed=["h:5d50f32ce404d5188168147983e3cb183cf519b3"]


---
## 3. The algebraic S(t): H(t) = (S(t), H(t-1), D, R, N)

`S(t)` is the union of the tree's leaves; `H(t-1)` is the union of the
previous tree's leaves. D/R/N fall out as lattice differences — and the
tree diff above gives the **exact** HLLSet-level version the union cannot.


In [8]:
let h_prev: HLLSet = HLLSet::union_all(vec![ha.clone(), hb.clone()]);
let s_now: HLLSet = HLLSet::union_all(vec![ha.clone(), hc.clone()]);

let departed: HLLSet = h_prev.difference(&s_now);
let retained: HLLSet = h_prev.intersection(&s_now);
let novel: HLLSet = s_now.difference(&h_prev);

println!("H(t) = (S(t), H(t-1), D, R, N)");
println!("  S(t)   popcount = {}", s_now.popcount());
println!("  H(t-1) popcount = {}", h_prev.popcount());
println!("  D = {}  R = {}  N = {}", departed.popcount(), retained.popcount(), novel.popcount());

assert_eq!(departed.union(&retained).popcount(), h_prev.popcount(), "D ∪ R = H(t-1)");
assert_eq!(retained.union(&novel).popcount(), s_now.popcount(), "R ∪ N = S(t)");
assert_eq!(departed.intersection(&novel).popcount(), 0, "D ∩ N = ∅");
println!("invariants hold: D∪R=H(t-1), R∪N=S(t), D∩N=∅");


H(t) = (S(t), H(t-1), D, R, N)


  S(t)   popcount = 6


  H(t-1) popcount = 6


  D = 2  R = 4  N = 2


invariants hold: D∪R=H(t-1), R∪N=S(t), D∩N=∅


---
## 4. The ewm-git link: commits pin S(t); the tree root links to the views

`ewm-git` already computes the same D/R/N per commit. The context tree adds
the missing piece: the working set as an addressable object whose root can
be referenced from the commit — linking persistence to the vocabulary views.


In [9]:
let mut repo: Repository<MemoryStore> = Repository::new(MemoryStore::default());
let root: ObjectId = repo.commit(&LatticeState::single(&h_prev), &[], "S(t-1)").unwrap();
let tip: ObjectId = repo.commit(&LatticeState::single(&s_now), &[root.clone()], "S(t)").unwrap();

let cv = view(&repo, &tip).unwrap();
println!("ewm-git CommitView: D={}  R={}  N={}",
    cv.departed.popcount(), cv.retained.popcount(), cv.new.popcount());

let s_now_key = h_key(&[0, 2, 3, 5, 8, 12]);
println!("committed S(t) content key: {}", s_now_key);
println!("context tree root:          {}", tree_now.root());
println!("=> the commit pins S(t); the tree root is the S(t) that links it to its views.");


ewm-git CommitView: D=2  R=4  N=2


committed S(t) content key: h:0684935c93f5940873b55a5323b361fac61a6623


context tree root:          c3eeb36c9e16e6f0efa8483c58c016db2ac86425


=> the commit pins S(t); the tree root is the S(t) that links it to its views.


---
## Summary

- `V_j(H_i) = M(H_i, L_j)` — views are function values with provenance `(h, l)`;
  the full image is the CRDT union over LUTs.
- The **context tree** gives the working set an exact, content-addressed
  identity: root changes iff the leaf set (or its views) changes; diff is
  exact at the HLLSet level.
- **S(t) is now algebraic**: the tree's union feeds the Noether equation
  `H(t) = (S(t), H(t-1), D, R, N)`, with invariants asserted.
- `ewm-git` commits pin S(t); the tree root is the bridge from persistence
  to the vocabulary views. (Storing the tree root in the commit object is
  the next prototype step.)
